In [1]:
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [2]:
from rasterio.crs import CRS

In [3]:
import os

In [4]:
import netCDF4
import rioxarray
import rasterio

In [5]:
base = os.path.join(os.getcwd(),'..')

In [6]:
data = os.path.join(base,'data(LPJmL)','new_run')

In [8]:
for r,d,f in os.walk(data):
    for fl in f:
        if fl.endswith('.nc') and 'MIRCA' in fl:
            print(fl)
            fn = os.path.join(r,fl)


cftfrac_5min_42bands_new_MIRCAOS_inclWallonie_reduced.nc


In [9]:
ls = ("temperate_cereals","rice","maize","tropical cereals","pulses","temperate roots",
             "tropical roots","sunflower","soybeans","groundnuts","rapeseed","sugarcane", 
      "barley","cotton","wheat2","rice2", "rice3","others","grasses","biofuels1","biofuels2")


In [10]:
# fn = os.path.join(r,'cft_evap.nc')
ds = netCDF4.Dataset(fn)
# nc_file = fn
# ds = xr.open_dataset(nc_file, decode_times=False)

In [11]:
ds

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    Created on: Thu Jul  3 11:17:30 2025
    dimensions(sizes): longitude(4320), latitude(2160), pft(42), time(1)
    variables(dimensions): float64 longitude(longitude), float64 latitude(latitude), int32 pft(pft), float64 time(time), float32 landfrac(time, pft, latitude, longitude)
    groups: 

In [12]:
var_name = 'landfrac'
evap = ds[var_name]    
# pft_names = ds["NamePFT"].values 
len(evap)

1

In [19]:
# fn = os.path.join(r,'cft_evap.nc')
nc_file = fn   
os.makedirs(os.path.join(base,'tiffs',var_name),exist_ok = True)
out_tif = os.path.join(base,'tiffs',var_name,var_name+'.tiff')
nodata = -1e+32
var_name = 'landfrac'
ds = xr.open_dataset(nc_file, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds[var_name]    
# pft_names = ds["NamePFT"].values  
ntime, bands, nlat, nlon = da.shape

lat = da["latitude"].values
lon = da["longitude"].values

da = da.squeeze()
da = da.sortby("latitude", ascending=True)

dlat = abs(lat[1] - lat[0])
dlon = abs(lon[1] - lon[0])


transform = from_origin(
    lon.min()-dlon/2,
    lat.max()+dlat/2,
    dlon,
    dlat
)


fill_value = da.attrs.get("_FillValue", nodata)


for t in range(evap.sizes["time"]):
    time_val = ds["time"].values[t]

    
    out_tif = os.path.join(base,'tiffs',var_name,var_name+f"_time_{t:03d}.tif")
    
    data = evap.isel(time=t).values.astype(np.float32)

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=nlat,
        width=nlon,
        count=bands,               
        dtype="float32",
        crs="""GEOGCS["WGS 84",
            DATUM["WGS_1984",
                SPHEROID["WGS 84",6378137,298.257223563]],
            PRIMEM["Greenwich",0],
            UNIT["degree",0.0174532925199433]]""",
        transform=transform,
        nodata=nodata,
        compress="lzw"
    ) as dst:

        
        for i in range(0,42):
            band = data[i, :, :]
            dst.write(band, i + 1)
            # dst.set_band_description(i + 1, str(pft_name))
            
    with rasterio.open(out_tif) as src:
        data = src.read()      
        profile = src.profile

    data_flipped = np.flip(data, axis=1)
    
    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(data_flipped)


    print(f"Written {out_tif}")


Written notebooks\..\tiffs\landfrac\landfrac_time_000.tif
